# Step 3 — Sentiment Classification

Run **XLM-RoBERTa** (`cardiffnlp/twitter-xlm-roberta-base-sentiment`) on every text:

| `text_type` | description |
|---|---|
| `reference_en` | Original English sentence from the dataset |
| `gpt_translated` | English translation produced by GPT-5.2-chat |
| `gemini_translated` | English translation produced by Gemini-2.5-flash |

Output → `outputs/classifications.csv`  
Columns: `id | source_lang | text_type | text | sentiment | confidence`

In [1]:
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

import warnings; warnings.filterwarnings("ignore")
import pandas as pd
from transformers import pipeline

DATA_PATH = r"c:\Users\Nguyen Ngo\Downloads\Thesis Quynh\data\sentences.csv"
GPT_PATH = r"c:\Users\Nguyen Ngo\Downloads\Thesis Quynh\outputs\translations_gpt.csv"
GEMINI_PATH = r"c:\Users\Nguyen Ngo\Downloads\Thesis Quynh\outputs\translations_gemini.csv"
OUT_PATH = r"c:\Users\Nguyen Ngo\Downloads\Thesis Quynh\outputs\classifications.csv"

sentences = pd.read_csv(DATA_PATH)
gpt_df = pd.read_csv(GPT_PATH)
gemini_df = pd.read_csv(GEMINI_PATH)

print(f"sentences : {len(sentences)} rows")
print(f"gpt transl. : {len(gpt_df)} rows")
print(f"gemini transl: {len(gemini_df)} rows")

sentences : 200 rows
gpt transl. : 533 rows
gemini transl: 533 rows


In [2]:
import torch

# ── Load classifier ────────────────────────────────────────────────────────────
device = 0 if torch.cuda.is_available() else -1
print(f"Using: {'GPU (CUDA)' if device == 0 else 'CPU'}  |  torch {torch.__version__}")

# 3-class: positive / neutral / negative
classifier = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    tokenizer="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    truncation=True,
    max_length=512,
    device=device,
    batch_size=32,
)

LABEL_MAP = {"positive": "positive", "neutral": "neutral", "negative": "negative"}

def classify(text: str) -> dict:
    if not isinstance(text, str) or text.startswith("ERROR"):
        return {"sentiment": None, "confidence": None}
    result = classifier(text)[0]
    label  = LABEL_MAP.get(result["label"].lower(), result["label"].lower())
    return {"sentiment": label, "confidence": round(result["score"], 4)}

print("Classifier loaded.")


Using: GPU (CUDA)  |  torch 2.6.0+cu118


Device set to use cuda:0


Classifier loaded.


In [3]:
# ── Build a flat list of all texts to classify ─────────────────────────────────
SOURCE_LANGS = ["Spanish", "French", "Italian"]

records = []

# 1) Reference English — one per source language
#    (same English sentence used as reference for each language pair)
for _, row in sentences.iterrows():
    for lang in SOURCE_LANGS:
        records.append({
            "id":          row["id"],
            "source_lang": lang,
            "text_type":   "reference_en",
            "text":        row["English"],
        })

# 2) GPT translations
for _, row in gpt_df.iterrows():
    records.append({
        "id":          row["id"],
        "source_lang": row["source_lang"],
        "text_type":   "gpt_translated",
        "text":        row["translated_text"],
    })

# 3) Gemini translations
for _, row in gemini_df.iterrows():
    records.append({
        "id":          row["id"],
        "source_lang": row["source_lang"],
        "text_type":   "gemini_translated",
        "text":        row["translated_text"],
    })

all_df = pd.DataFrame(records)
print(f"Total texts to classify: {len(all_df)}")
all_df["text_type"].value_counts()

Total texts to classify: 1666


text_type
reference_en         600
gpt_translated       533
gemini_translated    533
Name: count, dtype: int64

In [4]:
# ── Classify (with resume support) ────────────────────────────────────────────
if os.path.exists(OUT_PATH):
    done_df  = pd.read_csv(OUT_PATH)
    done_idx = set(zip(done_df["id"], done_df["source_lang"], done_df["text_type"]))
    print(f"Resuming — {len(done_df)} rows already classified")
else:
    done_df  = pd.DataFrame()
    done_idx = set()

new_rows = []
total = len(all_df)

for i, row in all_df.iterrows():
    key = (row["id"], row["source_lang"], row["text_type"])
    if key in done_idx:
        continue
    result = classify(row["text"])
    new_rows.append({**row.to_dict(), **result})

    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{total} classified ...")

new_df = pd.DataFrame(new_rows)
final  = pd.concat([done_df, new_df], ignore_index=True) if len(done_df) else new_df
final.to_csv(OUT_PATH, index=False, encoding="utf-8")
print(f"Done — {len(final)} rows saved → {OUT_PATH}")
final.head()

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  100/1666 classified ...
  200/1666 classified ...
  300/1666 classified ...
  400/1666 classified ...
  500/1666 classified ...
  600/1666 classified ...
  700/1666 classified ...
  800/1666 classified ...
  900/1666 classified ...
  1000/1666 classified ...
  1100/1666 classified ...
  1200/1666 classified ...
  1300/1666 classified ...
  1400/1666 classified ...
  1500/1666 classified ...
  1600/1666 classified ...
Done — 1666 rows saved → c:\Users\Nguyen Ngo\Downloads\Thesis Quynh\outputs\classifications.csv


,id,source_lang,text_type,text,sentiment,confidence
0,1,Spanish,reference_en,Cable car gives quick access to spectacular vi...,positive,0.7136
1,1,French,reference_en,Cable car gives quick access to spectacular vi...,positive,0.7136
2,1,Italian,reference_en,Cable car gives quick access to spectacular vi...,positive,0.7136
3,2,Spanish,reference_en,A museum at the top in an old fort and also go...,positive,0.7086
4,2,French,reference_en,A museum at the top in an old fort and also go...,positive,0.7086


In [5]:
# ── Sanity check ───────────────────────────────────────────────────────────────
print(final.groupby(["text_type", "sentiment"])["id"].count().unstack(fill_value=0))
print(f"\nNull sentiments: {final['sentiment'].isna().sum()}")

sentiment          negative  neutral  positive
text_type                                     
gemini_translated       193       91       249
gpt_translated          190       90       251
reference_en            222      120       258

Null sentiments: 2
